# EPC Data Cleaning: Camden, London

Source: Domestic Energy Performance Certificates, Camden, Jan 2012 to Jun 2026.

This notebook turns the raw extract into a cleaned dataset, using the problems found in
the profiling notebook and ranked in the data quality report.

Approach: no rows are deleted. Every certificate is kept and marked with flags, so every
cleaning decision stays visible and can be reversed or disagreed with.


## 1. Load data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/camden_certificates.csv", low_memory=False)

df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
df['lodgement_date'] = pd.to_datetime(df['lodgement_date'], errors='coerce')

print(df.shape)
print("Unreadable inspection dates:", df['inspection_date'].isna().sum())

(102185, 93)
Unreadable inspection dates: 0


## 2. Which date decides the latest certificate?

A property can hold several certificates, so keeping the most recent one means sorting by
a date. There are two candidates: inspection_date, when the assessor actually visited, and
lodgement_date, when the certificate was filed.

Profiling found 1,100 certificates where lodgement_date falls before inspection_date, which
is impossible. Before choosing a date to trust, check how many properties that affects.

In [2]:
bad_dates = df[df['lodgement_date'] < df['inspection_date']]
print("Records where lodgement is before inspection:", len(bad_dates))

bad_uprns = bad_dates['uprn'].dropna().unique()
print("Distinct properties affected:", len(bad_uprns))

later_valid = df[
    (df['uprn'].isin(bad_uprns)) &
    (~df.index.isin(bad_dates.index))
]
print("Of those, properties that also hold another certificate:", later_valid['uprn'].nunique())

Records where lodgement is before inspection: 1100
Distinct properties affected: 942
Of those, properties that also hold another certificate: 605


942 properties hold a bad-date certificate, and 605 of them also hold another valid one,
so for those the bad record can lose out naturally. That leaves 337 properties where the
bad-date certificate is the only one on file.

Decision: sort by inspection_date. lodgement_date is the field proven wrong here, so
sorting on it would carry a known error straight into the cleaning step. inspection_date
also answers the real question better: when was this property last actually assessed.

## 3. What identifies a single home?

To keep the latest certificate per property, the notebook first needs a reliable way to say
"these two certificates describe the same home".

The obvious candidate is uprn, the unique property reference. But profiling showed one uprn
can cover up to 69 separate flats in a single building, so uprn alone would merge homes that
are genuinely different. Address alone is no safer, since different homes can share address
text and the same home is often typed several different ways.

Test both, and test them combined.

In [3]:
df['address_clean'] = (
    df['address'].astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

df[df['uprn'] == 5033858.0][['uprn', 'address', 'address_clean',
                             'inspection_date', 'current_energy_rating', 'total_floor_area']]

,uprn,address,address_clean,inspection_date,current_energy_rating,total_floor_area
22655,5033858.0,"Flat 5, 91 Dartmouth Park Hill",flat 5 91 dartmouth park hill,2022-08-15,D,48
79270,5033858.0,"Flat 5, 91, Dartmouth Park Hill",flat 5 91 dartmouth park hill,2020-09-11,D,40
94025,5033858.0,"FLAT 5, 91 DARTMOUTH PARK HILL",flat 5 91 dartmouth park hill,2021-05-26,D,46


In [4]:
has_uprn = df[df['uprn'].notna()]

print("Certificates with a uprn:      ", len(has_uprn))
print("Distinct uprn alone:           ", has_uprn['uprn'].nunique())
print("Distinct uprn + raw address:   ", len(has_uprn.drop_duplicates(subset=['uprn', 'address'])))
print("Distinct uprn + cleaned address:", len(has_uprn.drop_duplicates(subset=['uprn', 'address_clean'])))

Certificates with a uprn:       92883
Distinct uprn alone:            71513
Distinct uprn + raw address:    82941
Distinct uprn + cleaned address: 75745


Three numbers, three different answers to "how many homes are here".

uprn alone gives 71,513, which is too low, because building-level UPRNs collapse many flats
into one. Adding the raw address gives 82,941, but that is too high, because the same flat
typed three ways counts three times. Cleaning the address first gives 75,745, about 4,232
more homes than uprn alone would have found.

Cleaning means lowercasing, stripping punctuation, and collapsing repeated spaces. All three
matter: without the spacing step, an address written with a stray double space still fails to
match the same address written normally.

Decision: identify a home by uprn plus cleaned address. Neither part is safe on its own.
Clean the address before comparing, or real duplicates slip through as separate homes.

## 4. Can one address hold several UPRNs?

The key from Section 3 only treats two certificates as the same home when both the uprn and
the cleaned address match. That protects different flats that share a building-level uprn.

But it leaves the opposite case unhandled: if one address carries two different UPRNs, the
rule keeps both rows as separate homes. UPRNs are supposed to be unique per address, so this
is worth checking before trusting the rule.

In [5]:
uprns_per_address = has_uprn.groupby('address_clean')['uprn'].nunique()
print(uprns_per_address.sort_values(ascending=False).head(10))

address_clean
33 cantelowes road camden                      3
4 south villas camden                          3
307d finchley road                             2
307e finchley road                             2
15 spencer walk                                2
221 belsize road                               2
307f finchley road                             2
90 st augustines road camden                   2
35a buckland crescent                          2
flat 122 endsleigh court upper woburn place    2
Name: uprn, dtype: int64


In [6]:
df[df['address_clean'] == '33 cantelowes road camden'][
    ['uprn', 'uprn_source', 'address', 'inspection_date',
     'total_floor_area', 'property_type', 'current_energy_rating']
]

,uprn,uprn_source,address,inspection_date,total_floor_area,property_type,current_energy_rating
4204,5120033.0,Energy Assessor,"33 Cantelowes Road, Camden",2023-03-29,60,Flat,C
4665,5080666.0,Address Matched,"33 Cantelowes Road, Camden",2023-03-29,60,Flat,C
92575,5120032.0,Energy Assessor,"33 Cantelowes Road, Camden",2023-03-29,60,Flat,C


33 Cantelowes Road carries three UPRNs, all inspected on the same day, all recording 60 m2,
Flat, band C.

The uprn_source column explains most of it. Two of the three (5120032 and 5120033, consecutive
numbers) came from Energy Assessor, meaning a person on site supplied them, which is how
genuinely separate units get numbered. The third came from Address Matched, meaning no uprn
was supplied and one was attached afterwards by matching the address text.

So this is probably real separate flats plus one machine-matched artefact, but the data cannot
prove it either way. Identical floor area and rating could mean identical units in one block,
or one assessment recorded more than once.

Decision: do not delete on this evidence. Wrongly deleting a real home destroys data that
cannot be recovered, while wrongly keeping a duplicate only inflates a count. Flag these rows
instead, so the uncertainty stays visible.

Second decision: use uprn_source as a trust signal when breaking ties, since an
assessor-supplied uprn is stronger evidence than a machine-matched one.

## 5. Build the cleaned dataset

the sections above settled four questions. This section applies them.

No rows are deleted. Every certificate is kept and marked with flags, so the cleaning is
visible and reversible. Later analysis can then work on the latest record per property by filtering
on is_latest, and anyone who disagrees with a rule can re-derive their own version from the
same file.

Rules applied:
- A home is identified by uprn plus cleaned address (Section 3).
- Certificates with no uprn are keyed on cleaned address alone. That is weaker evidence, so
  they are counted separately in the checks below.
- Latest means latest inspection_date, not lodgement_date (Section 2).
- Ties are broken deterministically, preferring an assessor-supplied uprn over a
  machine-matched one (Section 4), so the notebook gives the same answer every run.
- Addresses carrying more than one uprn are flagged, not resolved (Section 4).

Columns added:
- property_key: the identifier two certificates must share to count as the same home
- is_latest: True for the most recent certificate of each home
- certs_for_property: how many certificates that home holds in total
- uprns_at_address: how many different UPRNs appear at that cleaned address
- same_address_multi_uprn: True where that count is above one
- identical_diff_uprn: same address, different uprn, and identical measured values
- uprn_present: whether the certificate carries a uprn at all
- uprn_trusted: whether the uprn was supplied by the assessor rather than matched by a system

In [7]:
work = df.copy()

work['uprn_present'] = work['uprn'].notna()
work['uprn_trusted'] = work['uprn_source'].eq('Energy Assessor')

work['property_key'] = np.where(
    work['uprn_present'],
    work['uprn'].astype('Int64').astype(str) + '|' + work['address_clean'],
    'NOUPRN|' + work['address_clean']
)

work = work.sort_values(
    ['property_key', 'inspection_date', 'lodgement_date', 'uprn_trusted', 'certificate_number'],
    ascending=[True, False, False, False, False],
    na_position='last'
)

work['is_latest'] = ~work.duplicated(subset='property_key', keep='first')
work['certs_for_property'] = work.groupby('property_key')['certificate_number'].transform('size')

uprns_per_address = work.loc[work['uprn_present']].groupby('address_clean')['uprn'].nunique()
work['uprns_at_address'] = work['address_clean'].map(uprns_per_address)
work['same_address_multi_uprn'] = work['uprns_at_address'] > 1

twin_cols = ['address_clean', 'inspection_date', 'total_floor_area',
             'current_energy_rating', 'property_type']
work['identical_diff_uprn'] = (
    work['same_address_multi_uprn'] & work.duplicated(subset=twin_cols, keep=False)
)

print("Total rows kept:                    ", len(work))
print("Marked as latest per home:          ", work['is_latest'].sum())
print("  with a uprn:                      ", (work['is_latest'] & work['uprn_present']).sum())
print("  without a uprn:                   ", (work['is_latest'] & ~work['uprn_present']).sum())
print("Rows at addresses with 2+ uprns:    ", work['same_address_multi_uprn'].sum())
print("Rows flagged as possible resubmits: ", work['identical_diff_uprn'].sum())

Total rows kept:                     102185
Marked as latest per home:           83675
  with a uprn:                       75745
  without a uprn:                    7930
Rows at addresses with 2+ uprns:     514
Rows flagged as possible resubmits:  19


In [8]:
noup_addr = work.loc[~work['uprn_present'] & work['is_latest'], 'address_clean']
hasup_addr = set(work.loc[work['uprn_present'], 'address_clean'])

overlap = noup_addr.isin(hasup_addr).sum()
print("No-uprn homes whose address also appears in the has-uprn group:", overlap)
print("Distinct keys marked latest:", work['is_latest'].sum())

No-uprn homes whose address also appears in the has-uprn group: 175
Distinct keys marked latest: 83675


is_latest marks the newest certificate per key, not per home. A home whose certificates are
split between having and not having a uprn produces two keys, so it gets marked twice.

175 of the no-uprn homes sit at an address that also appears in the has-uprn group, so up to
175 homes are counted twice. That puts the real figure between 83,500 and 83,675.

Treat 83,675 as an upper bound on homes, not a count of them. The gap is small but it is not
zero, and it exists because 9.1% of certificates arrive with no uprn in the first place.

## 6. Validate

Three checks. The first two are assertions, so the notebook fails loudly rather than passing
bad data downstream. The third is a spot check on a case examined by hand in Section 3.

In [9]:
assert len(work) == len(df), "Row count changed, rows were lost or duplicated"
assert work.groupby('property_key')['is_latest'].sum().max() == 1, "A home has more than one latest certificate"
assert (work['is_latest'] & work['uprn_present']).sum() == 75745, "Does not match the Section 3 count"

print("All checks passed.")
print()
print("Spot check, Dartmouth Park Hill (three certificates, one flat):")
work[work['uprn'] == 5033858.0][
    ['uprn', 'address', 'inspection_date', 'is_latest', 'certs_for_property']
]

All checks passed.

Spot check, Dartmouth Park Hill (three certificates, one flat):


,uprn,address,inspection_date,is_latest,certs_for_property
22655,5033858.0,"Flat 5, 91 Dartmouth Park Hill",2022-08-15,True,3
94025,5033858.0,"FLAT 5, 91 DARTMOUTH PARK HILL",2021-05-26,False,3
79270,5033858.0,"Flat 5, 91, Dartmouth Park Hill",2020-09-11,False,3


## 7. Normalise construction_age_band

The construction age of a property is recorded in construction_age_band, but the column
does not use one format. Some records hold a descriptive band such as
"England and Wales: 1900-1929", others hold a bare year such as "2017".

Two formats in one column means the values cannot be compared or sorted as they stand.
Before converting anything, look at every value the column actually holds.

In [10]:
print("Distinct values:", df['construction_age_band'].nunique(dropna=False))

Distinct values: 58


In [11]:
print(df['construction_age_band'].value_counts(dropna=False))

construction_age_band
England and Wales: 1900-1929       26532
England and Wales: before 1900     21094
England and Wales: 1930-1949       11662
England and Wales: 1950-1966        8410
England and Wales: 1967-1975        7678
England and Wales: 1976-1982        3514
England and Wales: 1983-1990        2290
England and Wales: 1996-2002        1995
England and Wales: 2003-2006        1928
England and Wales: 2007-2011        1800
England and Wales: 1991-1995        1375
England and Wales: 2012 onwards     1273
2017                                1135
2014                                1032
2015                                1009
2021                                 994
2016                                 973
2013                                 953
2019                                 931
2018                                 731
2012                                 654
2024                                 541
England and Wales: 2007 onwards      495
2023                               

The column holds two formats: 15 descriptive bands covering about 90,000 records, and 42
bare years covering about 12,000. Three are blank.

The bands are also not one consistent set. "2007-2011" and "2007 onwards" both appear, as do
"2012 onwards", "2012-2021" and "2022 onwards". Open-ended and closed bands covering the same
start year cannot belong to the same vocabulary, so this column pools several generations of
assessment software.

Decision: convert to a start year and an end year rather than a single number, plus a note of
which format each value came from. A single number would make a 30 year band look as precise
as an exact year. "before 1900" has no start year and "2012 onwards" has no end year, so those
stay empty rather than being filled with a guess.

In [12]:
core = (df['construction_age_band']
        .astype('string')
        .str.replace(r'^England and Wales:\s*', '', regex=True)
        .str.strip())

year_from = pd.Series(pd.NA, index=df.index, dtype='Int64')
year_to   = pd.Series(pd.NA, index=df.index, dtype='Int64')

m = core.str.match(r'^before \d{4}$', na=False)
year_to[m] = core[m].str.extract(r'(\d{4})')[0].astype(int) - 1

m = core.str.match(r'^\d{4}-\d{4}$', na=False)
ext = core[m].str.extract(r'^(\d{4})-(\d{4})$')
year_from[m] = ext[0].astype(int)
year_to[m]   = ext[1].astype(int)

m = core.str.match(r'^\d{4} onwards$', na=False)
year_from[m] = core[m].str.extract(r'(\d{4})')[0].astype(int)

m = core.str.match(r'^\d{4}$', na=False)
year_from[m] = core[m].astype(int)
year_to[m]   = core[m].astype(int)

df['construction_year_from'] = year_from
df['construction_year_to']   = year_to
df['construction_age_precision'] = np.where(
    core.str.match(r'^\d{4}$', na=False), 'exact year',
    np.where(core.isna(), 'missing', 'band')
)

unconverted = (df['construction_age_band'].notna()
               & df['construction_year_from'].isna()
               & df['construction_year_to'].isna())
print("Values that failed to convert:", unconverted.sum())
print()
print(df['construction_age_precision'].value_counts())

Values that failed to convert: 0

construction_age_precision
band          90439
exact year    11743
missing           3
Name: count, dtype: int64


In [13]:
work['construction_year_from'] = df['construction_year_from']
work['construction_year_to'] = df['construction_year_to']
work['construction_age_precision'] = df['construction_age_precision']

print("Columns now on work:", work.shape[1])

Columns now on work: 105


Every value converted, with none left unparsed. 90,439 records carry a band, 11,743 carry an
exact year, and 3 are blank.

The three new columns are copied onto the working dataset, since it was built before this
step ran.

## 8. Flag possible rebuilds

A building cannot be built twice, so the construction year recorded for a home should be the
same on every certificate it holds. Profiling found a Hampstead building where it was not: the
same flat addresses appear in 2016 described as built before 1900, and again in 2018 described
as built in 2017, with the floor areas and ratings changing too. The building was demolished
and rebuilt, and the register has no field marking the older records as superseded.

Now that construction age is comparable, these cases can be found systematically. Start by
counting how many homes hold certificates that disagree about when the property was built.

In [14]:
work['construction_year_est'] = (
    work['construction_year_from'].fillna(work['construction_year_to'])
)

print("Records with no usable construction year:",
      work['construction_year_est'].isna().sum())

year_span = (work.loc[work['construction_year_est'].notna()]
                 .groupby('property_key')['construction_year_est']
                 .agg(['nunique', 'min', 'max']))

disagree = year_span[year_span['nunique'] > 1].copy()
disagree['gap'] = disagree['max'] - disagree['min']

print("Homes whose certificates disagree on construction year:", len(disagree))
print()
print(disagree['gap'].describe())

Records with no usable construction year: 3
Homes whose certificates disagree on construction year: 9261

count       9261.0
mean     23.653061
std      29.334973
min            1.0
25%            1.0
50%           17.0
75%           30.0
max          158.0
Name: gap, dtype: Float64


In [15]:
bins = [0, 2, 5, 10, 20, 30, 50, 75, 100, 200]
print("Homes by size of disagreement:")
print(pd.cut(disagree['gap'], bins=bins).value_counts().sort_index())
print()
print("Widest 10 disagreements:")
print(disagree.sort_values('gap', ascending=False).head(10))

Homes by size of disagreement:
gap
(0, 2]        2757
(2, 5]         652
(5, 10]        856
(10, 20]      1392
(20, 30]      1437
(30, 50]      1048
(50, 75]       257
(75, 100]      361
(100, 200]     501
Name: count, dtype: int64

Widest 10 disagreements:
                                                    nunique   min   max  gap
property_key                                                                
5192062|68 cambridge terrace                              2  1825  1983  158
5078919|flat 1 140a camden high street                    3  1899  2026  127
5078920|flat 2 140a camden high street                    2  1899  2026  127
5078921|flat 3 140a camden high street                    2  1899  2026  127
5084546|flat 7 south hill mansions 6870 south h...        2  1899  2025  126
5103155|flat 5 7 adamson road                             2  1899  2024  125
5103156|flat 2 7 adamson road                             2  1899  2024  125
5103153|flat 6 7 adamson road                    

In [16]:
cols = ['address', 'inspection_date', 'construction_age_band',
        'construction_year_est', 'total_floor_area', 'current_energy_rating']

print("140a Camden High Street:")
print(work[work['address_clean'].str.contains('140a camden high street', na=False)]
      .sort_values(['address_clean', 'inspection_date'])[cols]
      .to_string())

140a Camden High Street:
                                       address inspection_date           construction_age_band  construction_year_est  total_floor_area current_energy_rating
79075         Flat 1, 140a, Camden High Street      2015-09-18  England and Wales: before 1900                   1899               154                     D
71322         Flat 1, 140a, Camden High Street      2020-01-15    England and Wales: 1930-1949                   1930               107                     C
40099          Flat 1, 140A Camden High Street      2026-04-14                            2026                   2026                70                     C
7355           Flat 1, 140A Camden High Street      2026-04-14                            2026                   2026                70                     C
9175          Flat 2, 140a, Camden High Street      2015-10-21  England and Wales: before 1900                   1899                95                     D
99230          Flat 2, 140a

In [17]:
prop_stats = (work.loc[work['construction_year_est'].notna()]
                  .groupby('property_key')
                  .agg(year_min=('construction_year_est', 'min'),
                       year_max=('construction_year_est', 'max'),
                       area_min=('total_floor_area', 'min'),
                       area_max=('total_floor_area', 'max')))

prop_stats['year_gap'] = prop_stats['year_max'] - prop_stats['year_min']
prop_stats['area_change_pct'] = (
    (prop_stats['area_max'] - prop_stats['area_min']) / prop_stats['area_min'] * 100
)

rebuild_keys = prop_stats[
    (prop_stats['year_gap'] >= 30) & (prop_stats['area_change_pct'] >= 20)
].index

work['possible_rebuild'] = work['property_key'].isin(rebuild_keys)

print("Homes flagged as possible rebuilds:", len(rebuild_keys))
print("Certificates affected:", work['possible_rebuild'].sum())
print()
print("Check the known case, 140a Camden High Street:")
print(work.loc[work['address_clean'].str.contains('140a camden high street', na=False),
               'possible_rebuild'].value_counts())

Homes flagged as possible rebuilds: 693
Certificates affected: 1574

Check the known case, 140a Camden High Street:
possible_rebuild
True     10
False     9
Name: count, dtype: int64


In [18]:
suspect = (work.loc[work['possible_rebuild']]
               .groupby('property_key')['total_floor_area']
               .agg(['min', 'max']))

odd = suspect[(suspect['min'] < 10) | (suspect['max'] > 500)]

print("Flagged rebuild homes with an implausible floor area:", len(odd))
print("Share of the 693:", round(len(odd) / len(suspect) * 100, 1), "%")
print()
print(odd.head(10))

Flagged rebuild homes with an implausible floor area: 23
Share of the 693: 3.3 %

                                                    min   max
property_key                                                 
200096777|25 ranulf road                            402   805
5008259|21 lyndhurst road                           516   627
5010667|123 broadhurst gardens                      354   509
5019248|14 rosecroft avenue                         393   512
5037269|16 avenue road                              738  1752
5039253|63 frognal                                  551   853
5045169|21 belsize park                             494   655
5045873|flat 53 cecil rhodes house goldington s...    8    81
5047060|62 avenue road                              899  1341
5053361|41 frognal                                  337  1645


In [19]:
too_small = suspect[suspect['min'] < 10]
too_large = suspect[suspect['max'] > 500]

print("Flagged rebuild homes with a floor area under 10 m2:", len(too_small))
print("Flagged rebuild homes with a floor area over 500 m2:", len(too_large))
print()
print("The under-10 cases:")
print(too_small)

Flagged rebuild homes with a floor area under 10 m2: 8
Flagged rebuild homes with a floor area over 500 m2: 15

The under-10 cases:
                                                    min  max
property_key                                                
5045873|flat 53 cecil rhodes house goldington s...    8   81
5061986|flat 91 hillsborough court mortimer cre...    5   44
5122795|flat 6 186a finchley road                     8   12
5142524|flat a6 9 langtry walk                        8   21
5142528|flat a10 9 langtry walk                       9   21
5158211|flat 1 251 camden high street                 8   55
5175140|flat 7 72 shoot up hill                       6   56
5193693|flat 3 1 elm terrace constantine road         6   42


9,261 homes hold certificates that disagree about construction year, but most of that is
noise. A quarter disagree by a single year, and the common gaps of 17 and 30 years match
the distance between adjacent bands, so they are two assessors choosing neighbouring bands
for the same old house rather than any real change.

Requiring the floor area to change as well separates a rebuild from a recording difference.
Flagging homes with a construction year gap of 30 years or more and a floor area change of
20% or more leaves 693 homes across 1,574 certificates.

140a Camden High Street shows why the floor area matters. Flat 1 runs 154 m2 in 2015 recorded
as before 1900, then 107 m2 in 2020, then 70 m2 in 2026 recorded as built 2026. Flat 3 goes
from 62 m2 band E to 39 m2 band C. Every flat shrinks and every rating improves, all reassessed
on one day. The building was replaced with more, smaller, better insulated units.

Both thresholds are judgement calls, stated here so they can be challenged. The 30 year gap is
wide enough to clear adjacent band boundaries, and 20% is a floor area change too large to come
from measurement differences.

Limitation: the flag compares a home against its own history, so it only catches homes that
have one. At 140a, flats 5, 6 and 7 appear only in 2026 and are not flagged, because a new unit
in a rebuilt building has no earlier record to contradict. The flag marks contradictory records,
not every property in a rebuilt building.

Checked whether the flag is contaminated by the impossible floor areas found in profiling.
Of the 693 flagged homes, 8 hold a record under 10 m2 rising to a normal size, which is a bad
record paired with a good one rather than a property that changed. A further 15 exceed 500 m2,
but those are Frognal and Avenue Road houses that are genuinely large, so they are correctly
flagged rather than errors.

Contamination is therefore about 1.2%, and the flag stands as it is.

## 9. Standardise transaction_type

transaction_type records why a certificate was produced, such as a sale or a new build.
Profiling found the same value written more than one way: "Marketed sale" and "marketed sale"
differ only by capital letter, and "non marketed sale" and "non-marketed sale" only by a
hyphen. A filter looking for one spelling silently drops the records written the other way.

Unlike floor area or emissions, this is fixable rather than a guess. The intended value is
obvious, so the variants can be folded together without inventing anything.

Look at the raw values first.

In [20]:
print("Distinct values:", work['transaction_type'].nunique(dropna=False))
print()
print(work['transaction_type'].value_counts(dropna=False))

Distinct values: 21

transaction_type
Rental                       53680
Marketed sale                25177
New dwelling                 10572
None of the above             3474
Non-marketed sale             1761
marketed sale                 1408
rental (private)              1399
Assessment for Green Deal     1378
Stock condition survey         946
ECO assessment                 942
rental (social)                454
new dwelling                   388
NaN                            268
FiT application                 91
Grant scheme                    64
Following Green Deal            54
non marketed sale               53
RHI application                 32
not sale or rental              19
Re-mortgaging                   14
Non-grant scheme                11
Name: count, dtype: int64


In [21]:
work['transaction_type_clean'] = (
    work['transaction_type']
    .astype('string')
    .str.lower()
    .str.strip()
    .str.replace(r'\s*-\s*', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

print("Before:", work['transaction_type'].nunique(dropna=False))
print("After: ", work['transaction_type_clean'].nunique(dropna=False))
print()
print(work['transaction_type_clean'].value_counts(dropna=False))

Before: 21
After:  18

transaction_type_clean
rental                       53680
marketed sale                26585
new dwelling                 10960
none of the above             3474
non marketed sale             1814
rental (private)              1399
assessment for green deal     1378
stock condition survey         946
eco assessment                 942
rental (social)                454
<NA>                           268
fit application                 91
grant scheme                    64
following green deal            54
rhi application                 32
not sale or rental              19
re mortgaging                   14
non grant scheme                11
Name: count, dtype: Int64


21 values become 18. Three pairs merge: "Marketed sale" with "marketed sale", "New dwelling"
with "new dwelling", and "Non-marketed sale" with "non marketed sale". The counts confirm the
merges, with marketed sale going from 25,177 to 26,585 and non marketed sale from 1,761 to
1,814.

The 18 remaining values contain no misspellings. Every variant differed only by capital letter
or hyphen, which points to software dropdown lists that changed between versions rather than
anything typed by hand.

rental (private) and rental (social) are deliberately left separate from rental. They hold
1,853 records, all lodged before 2013, and they are the only tenure detail the register still
carries after it stopped recording the distinction. Folding them into rental for tidiness would
destroy exactly the information this audit reports as missing.

The original column is kept. transaction_type_clean sits alongside it, so the change is visible
rather than applied in place.

## 10. Flag implausible and duplicated records

Profiling found values that cannot be true: homes as small as 3 m2, negative CO2 emissions,
and certificates lodged before the inspection that produced them. None of these can be
corrected, since the true value is unknowable, but they can be marked so that analysis can
exclude them deliberately rather than by accident.

Floor area gets two separate flags, not one. A home under 10 m2 is impossible. A home over
500 m2 is unusual but often real, since Camden holds genuinely large houses in Frognal and
Avenue Road. Pooling them into a single "implausible" flag would label real houses as errors,
which the rebuild check in Section 8 already demonstrated.

Same-day duplicate lodgements are also marked. is_latest already resolves them by keeping one,
but without a flag the finding would vanish from the cleaned file.

In [22]:
work['floor_area_implausible'] = work['total_floor_area'] < 10
work['floor_area_very_large']  = work['total_floor_area'] > 500

work['negative_emissions'] = work['co2_emissions_current'] < 0
work['negative_energy']    = work['energy_consumption_current'] < 0

work['lodged_before_inspection'] = work['lodgement_date'] < work['inspection_date']

work['same_day_duplicate'] = work.duplicated(
    subset=['property_key', 'lodgement_date'], keep=False
)

flag_cols = ['floor_area_implausible', 'floor_area_very_large',
             'negative_emissions', 'negative_energy',
             'lodged_before_inspection', 'same_day_duplicate']

print("Certificates carrying each flag:")
for c in flag_cols:
    print(f"  {c:26} {work[c].sum():>6}")

work['any_quality_flag'] = work[flag_cols].any(axis=1)

print()
print("Certificates with at least one flag:", work['any_quality_flag'].sum())
print("Of those, marked is_latest:", (work['any_quality_flag'] & work['is_latest']).sum())

Certificates carrying each flag:
  floor_area_implausible         82
  floor_area_very_large         306
  negative_emissions             27
  negative_energy                37
  lodged_before_inspection     1100
  same_day_duplicate            985

Certificates with at least one flag: 2493
Of those, marked is_latest: 1158


Five of the six counts match profiling exactly: 82 homes under 10 m2, 306 over 500 m2,
27 with negative emissions, 37 with negative energy, and 1,100 lodged before inspection.
The two notebooks reach the same figures by different routes, which confirms neither has
drifted.

same_day_duplicate returns 985 against the 927 rows profiling found in duplicate groups.
The extra 58 are pairs whose addresses differ only by punctuation or casing, which raw
address matching missed and the cleaned address catches.

2,493 certificates carry at least one flag, and 1,158 of those are marked is_latest. That
last figure matters: filtering on is_latest alone would carry over a thousand known-bad
records into any analysis, so the flags need to be applied as well.

## 11. Save

The cleaned file keeps every original row and column, plus the columns added by this notebook.
Nothing is overwritten: each added column sits alongside the one it came from, so the effect
of every cleaning step can be inspected against the original.

Analysis works on the latest certificate per property by filtering on is_latest, and should
apply the quality flags as well. 1,158 certificates are both marked is_latest and carry at
least one flag, so is_latest alone is not enough:

    clean = work[work['is_latest'] & ~work['any_quality_flag']]

The output goes to data/processed/, which is not committed to the repository, since the source
data is licensed and not redistributed here.

Columns added, identifying a home:
- address_clean: lowercased, punctuation stripped, repeated spaces collapsed
- uprn_present: whether the certificate carries a uprn at all
- uprn_trusted: whether the uprn was supplied by the assessor rather than matched by a system
- property_key: the identifier two certificates must share to count as the same home
- is_latest: the most recent certificate for each key
- certs_for_property: how many certificates that key holds

Columns added, uncertainty about identity:
- uprns_at_address: how many different uprns appear at that cleaned address
- same_address_multi_uprn: True where that count is above one
- identical_diff_uprn: same address, different uprn, identical measured values

Columns added, construction age:
- construction_year_from, construction_year_to: the age band converted to years
- construction_age_precision: whether the source was a band, an exact year, or missing
- construction_year_est: a single comparable year, using year_to where year_from is empty
- possible_rebuild: contradictory records on both construction year and floor area

Columns added, categories:
- transaction_type_clean: casing and hyphen variants folded together

Columns added, quality flags:
- floor_area_implausible: floor area under 10 m2, which is not a possible dwelling
- floor_area_very_large: floor area over 500 m2, unusual but often genuine in Camden
- negative_emissions: current CO2 emissions below zero
- negative_energy: current energy consumption below zero
- lodged_before_inspection: certificate filed before the inspection that produced it
- same_day_duplicate: the same home certificated more than once on one day
- any_quality_flag: True if any of the six flags above is set

In [23]:
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "camden_certificates_cleaned.csv"
work.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("Rows:", len(work))
print("Columns:", work.shape[1])

Saved to: ..\data\processed\camden_certificates_cleaned.csv
Rows: 102185
Columns: 115
